In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


# ============================================
# 1. Swiss roll 数据生成 + 高维嵌入
# ============================================

def generate_swiss_roll(n_samples, D=32, noise=0.0, device="cpu"):
    """
    生成 Swiss roll:
      - 参数 (u, v)
      - 3D: (x1, x2, x3)
      - 高维嵌入: A @ x_3d, A ∈ R^{D x 3} 正交列

    返回:
      X_high: (n_samples, D)
      y:      (n_samples,)  -- 一个光滑的 target（这里用 sin(u) 做 toy）
      u, v:   (n_samples,)  -- 方便你以后做 intrinsic 对比
    """
    u = torch.empty(n_samples, device=device).uniform_(3 * math.pi, 9 * math.pi)
    v = torch.empty(n_samples, device=device).uniform_(0.0, 20.0)

    x1 = u * torch.cos(u)
    x2 = v
    x3 = u * torch.sin(u)

    X3 = torch.stack([x1, x2, x3], dim=1)  # (n, 3)

    if noise > 0:
        X3 = X3 + noise * torch.randn_like(X3)

    # 随机高维正交嵌入
    # A: (D, 3), 列正交
    A = torch.randn(D, 3, device=device)
    # QR 分解，取 Q 的前 3 列
    Q, _ = torch.linalg.qr(A, mode="reduced")  # Q: (D, 3)
    X_high = X3 @ Q.T  # (n, 3) @ (3, D) -> (n, D)

    # toy 的回归目标：y = sin(u) / u 之类的平滑函数
    y = torch.sin(u) / (u + 1.0)

    return X_high, y, u, v


# ============================================
# 2. Hilbert 射影度量（正锥 R^m_{>0}）
# ============================================

def hilbert_distance(x, y, eps=1e-8):
    """
    x, y: (m,) 正向量（>0）
    d_H(x, y) = log( max_i x_i / y_i ) - log( min_i x_i / y_i )

    返回: float
    """
    # 避免 0
    x_safe = x.clamp_min(eps)
    y_safe = y.clamp_min(eps)
    ratio = x_safe / y_safe
    max_r = ratio.max()
    min_r = ratio.min()
    return (max_r.log() - min_r.log()).item()


# ============================================
# 3. 简单正锥模型：正权重线性回归
#    w = softplus(theta) ∈ R^D_{>0}
# ============================================

class ReLUPositiveLinear(nn.Module):
    def __init__(self, D):
        super().__init__()
        # 用小正数初始化，避免一开始就死在 ReLU 的 0 上
        theta0 = 0.1 * torch.ones(D)
        self.theta = nn.Parameter(theta0)

    def forward(self, X):
        w = F.relu(self.theta)
        return X @ w

    def relu_params_vector(self):
        return F.relu(self.theta).detach().clone()








